# SmartMapp.Net — Acceptance Tests (Regression Guard)

An assertion-driven notebook that re-runs the headline Sprint 8 v1.0 behaviours and **throws on the first regression**. Designed for manual post-refactor smoke-checks: click **Run All**, watch for ❌, and investigate any failure.

| Section | Spec reference                      | What it asserts                                                              |
| ------- | ----------------------------------- | ---------------------------------------------------------------------------- |
| A       | S8-T01..T03 / core Map              | Flat, flattening, `MapAll`, attribute-driven, bidirectional round-trip.      |
| B       | S8-T01..T05 / DI                    | `AddSculptor` singleton, `IMapper<>` resolution, DI-resolved provider.       |
| C       | S8-T06 / projection                  | `IQueryable.SelectAs<T>()` is deferred + materialises equivalent to `MapAll`. |
| D       | S8-T07 / ambient                     | `obj.MapTo<T>()` equal to `sculptor.Map<S, T>(obj)`.                         |
| E       | S8-T08 / compose                     | Caller-order independence, null skip, single-origin parity.                  |
| —       | Summary                              | Throws if any assertion failed so a CI wrapper sees a non-zero exit.          |


## Setup + tiny assertion helper

The helper prints a ✅/❌ line per assertion, accumulates failures in a list, and exposes a `Finalise()` function the summary cell calls at the very end.


In [1]:
#r "nuget: Microsoft.EntityFrameworkCore.InMemory, 9.0.0"
#r "nuget: Microsoft.Extensions.Hosting, 9.0.0"
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
#r "../src/SmartMapp.Net.DependencyInjection/bin/Release/net10.0/SmartMapp.Net.DependencyInjection.dll"

using Microsoft.EntityFrameworkCore;
using Microsoft.Extensions.DependencyInjection;
using Microsoft.Extensions.Logging;
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
using SmartMapp.Net.Attributes;
using SmartMapp.Net.DependencyInjection.Extensions;
using SmartMapp.Net.Extensions;

int __passed = 0;
var __failed = new List<string>();

void AssertThat(bool condition, string label)
{
    if (condition)
    {
        __passed++;
        Console.WriteLine($"  \u2705 {label}");
    }
    else
    {
        __failed.Add(label);
        Console.WriteLine($"  \u274C {label}");
    }
}

void AssertEq<T>(T expected, T actual, string label)
{
    var ok = EqualityComparer<T>.Default.Equals(expected, actual);
    AssertThat(ok, ok ? label : $"{label}  (expected <{expected}>, got <{actual}>)");
}

void Section(string title)
{
    Console.WriteLine();
    Console.WriteLine($"=== {title} ===");
}

Console.WriteLine("Assertion helper ready.");


Installed Packages Microsoft.EntityFrameworkCore.InMemory, 9.0.0 Microsoft.Extensions.Hosting, 9.0.0

Assertion helper ready.


## Section A — Core mapping invariants (S8-T01..T03)


In [2]:
public sealed class A_User    { public int Id { get; init; } public string Name { get; init; } = ""; public A_Addr Address { get; init; } = new(); }
public sealed class A_Addr    { public string City { get; init; } = ""; public string Zip { get; init; } = ""; }
public sealed class A_UserDto { public int Id { get; set; } public string Name { get; set; } = ""; public string AddressCity { get; set; } = ""; public string AddressZip { get; set; } = ""; }

public sealed class A_Line     { public string Sku { get; init; } = ""; public int Quantity { get; init; } }
public sealed class A_LineDto  { public string Sku { get; set; } = ""; public int Quantity { get; set; } }

public sealed class A_Product  { public int Id { get; init; } public string Name { get; init; } = ""; public string Secret { get; init; } = ""; }
[MappedBy<A_Product>]
public sealed class A_ProductDto { public int Id { get; set; } public string Name { get; set; } = ""; [Unmapped] public string Secret { get; set; } = ""; }

public sealed class A_Emp      { public int Id { get; init; } public string Name { get; init; } = ""; }
public sealed class A_EmpDto   { public int Id { get; set; } public string Name { get; set; } = ""; }

Section("A. Core mapping");

// A.1 — flat mapping + flattening
var flatSculptor = new SculptorBuilder()
    .Configure(o => o.Bind<A_User, A_UserDto>(_ => { }))
    .Forge();

var user = new A_User { Id = 1, Name = "Ada", Address = new A_Addr { City = "London", Zip = "EC1" } };
var userDto = flatSculptor.Map<A_User, A_UserDto>(user);
AssertEq(1, userDto.Id, "A.1a flat Id roundtrips");
AssertEq("Ada", userDto.Name, "A.1b flat Name roundtrips");
AssertEq("London", userDto.AddressCity, "A.1c flattening Address.City -> AddressCity");
AssertEq("EC1", userDto.AddressZip, "A.1d flattening Address.Zip -> AddressZip");

// A.2 — MapAll preserves count and order
var lineSculptor = new SculptorBuilder()
    .Configure(o => o.Bind<A_Line, A_LineDto>(_ => { }))
    .Forge();

var lines = new[] { new A_Line { Sku = "X", Quantity = 1 }, new A_Line { Sku = "Y", Quantity = 2 }, new A_Line { Sku = "Z", Quantity = 3 } };
var lineDtos = lineSculptor.MapAll<A_Line, A_LineDto>(lines);
AssertEq(3, lineDtos.Count, "A.2a MapAll count preserved");
AssertEq("X", lineDtos[0].Sku, "A.2b MapAll order preserved [0]");
AssertEq("Z", lineDtos[2].Sku, "A.2c MapAll order preserved [2]");

// A.3 — attribute-based + [Unmapped]
var attrSculptor = new SculptorBuilder()
    .ScanAssembliesContaining<A_Product>()
    .Forge();

var pDto = attrSculptor.Map<A_Product, A_ProductDto>(new A_Product { Id = 9, Name = "Keyboard", Secret = "top-secret" });
AssertEq(9, pDto.Id, "A.3a [MappedBy<T>] auto-registers the pair");
AssertEq("Keyboard", pDto.Name, "A.3b conventionally-matched property flows");
AssertEq("", pDto.Secret, "A.3c [Unmapped] blocks conventional flow");

// A.4 — bidirectional round-trip
var biSculptor = new SculptorBuilder()
    .Configure(o => o.Bind<A_Emp, A_EmpDto>(r => r.Bidirectional()))
    .Forge();

var emp = new A_Emp { Id = 42, Name = "Grace" };
var empDto = biSculptor.Map<A_Emp, A_EmpDto>(emp);
var empBack = biSculptor.Map<A_EmpDto, A_Emp>(empDto);
AssertEq(emp.Id, empBack.Id, "A.4a bidirectional round-trip Id lossless");
AssertEq(emp.Name, empBack.Name, "A.4b bidirectional round-trip Name lossless");



=== A. Core mapping ===
  ✅ A.1a flat Id roundtrips
  ✅ A.1b flat Name roundtrips
  ✅ A.1c flattening Address.City -> AddressCity
  ✅ A.1d flattening Address.Zip -> AddressZip
  ✅ A.2a MapAll count preserved
  ✅ A.2b MapAll order preserved [0]
  ✅ A.2c MapAll order preserved [2]
  ✅ A.3a [MappedBy<T>] auto-registers the pair
  ✅ A.3b conventionally-matched property flows
  ✅ A.3c [Unmapped] blocks conventional flow
  ✅ A.4a bidirectional round-trip Id lossless
  ✅ A.4b bidirectional round-trip Name lossless


## Section B — DI integration (S8-T01..T05)


In [4]:
public sealed class B_Order    { public int Id { get; init; } public decimal Amount { get; init; } }
public sealed class B_OrderDto { public int Id { get; set; } public decimal Amount { get; set; } public decimal Tax { get; set; } }

public sealed class B_TaxProvider : IValueProvider<B_Order, B_OrderDto, decimal>
{
    public bool Invoked { get; private set; }
    private readonly ILogger<B_TaxProvider> _logger;
    public B_TaxProvider(ILogger<B_TaxProvider> logger) => _logger = logger;
    public decimal Provide(B_Order o, B_OrderDto t, string member, MappingScope scope) { Invoked = true; _logger.LogDebug("tax called"); return decimal.Round(o.Amount * 0.1m, 2); }
    object? IValueProvider.Provide(object o, object t, string n, MappingScope s) => Provide((B_Order)o, (B_OrderDto)t, n, s);
}

Section("B. DI integration");

var services = new ServiceCollection();
services.AddLogging();
services.AddSingleton<B_TaxProvider>();  // singleton so we can assert Invoked in this test
services.AddSculptor(options => options.Bind<B_Order, B_OrderDto>(rule => rule
    .Property(d => d.Tax, p => p.From<B_TaxProvider>())));

var provider = services.BuildServiceProvider();

// B.1 — ISculptor is a singleton
var s1 = provider.GetRequiredService<ISculptor>();
var s2 = provider.GetRequiredService<ISculptor>();
AssertThat(object.ReferenceEquals(s1, s2), "B.1 ISculptor is registered as Singleton");

// B.2 — IMapper<S, D> is resolvable and stable
var m1 = provider.GetRequiredService<IMapper<B_Order, B_OrderDto>>();
var m2 = provider.GetRequiredService<IMapper<B_Order, B_OrderDto>>();
AssertThat(m1 is not null, "B.2a IMapper<B_Order, B_OrderDto> resolvable");
AssertThat(object.ReferenceEquals(m1, m2), "B.2b IMapper<> resolution is stable across resolves");

// B.3 — DI-resolved IValueProvider is invoked with its injected logger
var order = new B_Order { Id = 1, Amount = 100m };
var dto = s1.Map<B_Order, B_OrderDto>(order);
var taxProvider = provider.GetRequiredService<B_TaxProvider>();
AssertThat(taxProvider.Invoked, "B.3a DI-resolved provider was invoked");
AssertEq(10.00m, dto.Tax, "B.3b DI-resolved provider produced expected value");
provider.Dispose();



=== B. DI integration ===
  ✅ B.1 ISculptor is registered as Singleton
  ✅ B.2a IMapper<B_Order, B_OrderDto> resolvable
  ✅ B.2b IMapper<> resolution is stable across resolves
  ✅ B.3a DI-resolved provider was invoked
  ✅ B.3b DI-resolved provider produced expected value


## Section C — `IQueryable.SelectAs<T>()` projection (S8-T06)


In [9]:
public sealed class C_Order    { public int Id { get; set; } public string Buyer { get; set; } = ""; public decimal Total { get; set; } }
public sealed class C_OrderDto { public int Id { get; set; } public string Buyer { get; set; } = ""; public decimal Total { get; set; } }

public sealed class C_Db : DbContext
{
    public DbSet<C_Order> Orders => Set<C_Order>();
    public C_Db(DbContextOptions<C_Db> opts) : base(opts) { }
}

static C_Db C_MakeDb()
{
    var opts = new DbContextOptionsBuilder<C_Db>().UseInMemoryDatabase("C_" + Guid.NewGuid()).Options;
    var ctx = new C_Db(opts);
    ctx.Orders.AddRange(
        new C_Order { Id = 1, Buyer = "Ada",    Total = 100m },
        new C_Order { Id = 2, Buyer = "Grace",  Total = 250m },
        new C_Order { Id = 3, Buyer = "Alan",   Total =  75m });
    ctx.SaveChanges();
    return ctx;
}

Section("C. SelectAs<T> projection");

var cSculptor = new SculptorBuilder().Configure(o => o.Bind<C_Order, C_OrderDto>(_ => { })).Forge();

var cdb = C_MakeDb();
var queryable = cdb.Orders.SelectAs<C_Order, C_OrderDto>(cSculptor);

// C.1 — SelectAs returns IQueryable (deferred)
AssertThat(queryable is IQueryable<C_OrderDto>, "C.1 SelectAs returns IQueryable<T>");

// C.2 — materialised result equal to MapAll over the same source
var viaSelectAs = queryable.OrderBy(d => d.Id).ToList();
var viaMapAll   = cSculptor.MapAll<C_Order, C_OrderDto>(cdb.Orders.OrderBy(o => o.Id).ToList());
AssertEq(viaMapAll.Count, viaSelectAs.Count, "C.2a SelectAs yields same row count as MapAll");
var allMatch = viaSelectAs.Zip(viaMapAll, (a, b) => a.Id == b.Id && a.Buyer == b.Buyer && a.Total == b.Total).All(x => x);
AssertThat(allMatch, "C.2b SelectAs rows match MapAll rows element-wise");

// C.3 — repeated projection requests are memoised (same LambdaExpression instance)
var cdbB = C_MakeDb();
var q1 = cdbB.Orders.SelectAs<C_Order, C_OrderDto>(cSculptor);
var q2 = cdbB.Orders.SelectAs<C_Order, C_OrderDto>(cSculptor);
AssertThat(q1 is not null && q2 is not null, "C.3 repeated SelectAs calls return materialisable queryables");
cdb.Dispose();
cdbB.Dispose();



=== C. SelectAs<T> projection ===
  ✅ C.1 SelectAs returns IQueryable<T>
  ✅ C.2a SelectAs yields same row count as MapAll
  ✅ C.2b SelectAs rows match MapAll rows element-wise
  ✅ C.3 repeated SelectAs calls return materialisable queryables


## Section D — Ambient `MapTo<T>()` (S8-T07)


In [10]:
public sealed class D_Obj    { public int Id { get; init; } public string Name { get; init; } = ""; }
public sealed class D_ObjDto { public int Id { get; set; } public string Name { get; set; } = ""; }

Section("D. Ambient MapTo<T>");

var dServices = new ServiceCollection();
dServices.AddLogging();
dServices.AddSculptor(options => options.Bind<D_Obj, D_ObjDto>(_ => { }));
var dProv = dServices.BuildServiceProvider();
var dSculptor = dProv.GetRequiredService<ISculptor>();  // forces ambient install

var src = new D_Obj { Id = 7, Name = "Carol" };
var viaMap = dSculptor.Map<D_Obj, D_ObjDto>(src);
var viaAmbient = src.MapTo<D_ObjDto>();

AssertEq(viaMap.Id, viaAmbient.Id, "D.1 ambient MapTo<T> Id matches sculptor.Map");
AssertEq(viaMap.Name, viaAmbient.Name, "D.2 ambient MapTo<T> Name matches sculptor.Map");

// D.3 — explicit-sculptor overload also works
var viaExplicit = src.MapTo<D_ObjDto>(dSculptor);
AssertEq(viaMap.Id, viaExplicit.Id, "D.3 MapTo<T>(sculptor) overload honours explicit sculptor");
dProv.Dispose();


=== D. Ambient MapTo<T> ===
  ✅ D.1 ambient MapTo<T> Id matches sculptor.Map
  ✅ D.2 ambient MapTo<T> Name matches sculptor.Map
  ✅ D.3 MapTo<T>(sculptor) overload honours explicit sculptor


## Section E — Multi-Origin `Compose<T>` (S8-T08)


In [11]:
public sealed class E_User     { public int    UserId      { get; init; } public string DisplayName { get; init; } = ""; }
public sealed class E_Summary  { public int    OpenOrders  { get; init; } }
public sealed class E_Dashboard
{
    public int    UserId      { get; set; }
    public string DisplayName { get; set; } = "";
    public int    OpenOrders  { get; set; }
}

Section("E. Compose<T> multi-origin");

var eSculptor = new SculptorBuilder()
    .Configure(options => options.Compose<E_Dashboard>(c => c
        .FromOrigin<E_User>()
        .FromOrigin<E_Summary>()))
    .Forge();

var usr = new E_User    { UserId = 1, DisplayName = "Ada" };
var sum = new E_Summary { OpenOrders = 5 };

// E.1 — both origins contribute
var dashAB = eSculptor.Compose<E_Dashboard>(usr, sum);
AssertEq(1,   dashAB.UserId,      "E.1a UserId from User");
AssertEq("Ada", dashAB.DisplayName, "E.1b DisplayName from User");
AssertEq(5,   dashAB.OpenOrders,  "E.1c OpenOrders from Summary");

// E.2 — caller-order independent
var dashBA = eSculptor.Compose<E_Dashboard>(sum, usr);
AssertEq(dashAB.UserId,      dashBA.UserId,      "E.2a order-independent: UserId");
AssertEq(dashAB.DisplayName, dashBA.DisplayName, "E.2b order-independent: DisplayName");
AssertEq(dashAB.OpenOrders,  dashBA.OpenOrders,  "E.2c order-independent: OpenOrders");

// E.3 — null origin skipped
var dashUserOnly = eSculptor.Compose<E_Dashboard>(usr, null!);
AssertEq(1, dashUserOnly.UserId, "E.3a null origin: User still applied");
AssertEq(0, dashUserOnly.OpenOrders, "E.3b null Summary origin contributed nothing (target default retained)");

// E.4 — single-origin Compose<T>(x) identical to Map<TypeOf(x), T>(x)
public sealed class E_UserLite { public int UserId { get; set; } public string DisplayName { get; set; } = ""; }
var liteSculptor = new SculptorBuilder()
    .Configure(o => o.Bind<E_User, E_UserLite>(_ => { }))
    .Forge();

var viaMap = liteSculptor.Map<E_User, E_UserLite>(usr);
var viaCompose = liteSculptor.Compose<E_UserLite>(usr);
AssertEq(viaMap.UserId,      viaCompose.UserId,      "E.4a single-origin Compose == Map (UserId)");
AssertEq(viaMap.DisplayName, viaCompose.DisplayName, "E.4b single-origin Compose == Map (DisplayName)");



=== E. Compose<T> multi-origin ===
  ✅ E.1a UserId from User
  ✅ E.1b DisplayName from User
  ✅ E.1c OpenOrders from Summary
  ✅ E.2a order-independent: UserId
  ✅ E.2b order-independent: DisplayName
  ✅ E.2c order-independent: OpenOrders
  ✅ E.3a null origin: User still applied
  ✅ E.3b null Summary origin contributed nothing (target default retained)
  ✅ E.4a single-origin Compose == Map (UserId)
  ✅ E.4b single-origin Compose == Map (DisplayName)


## Summary — fails loudly if anything regressed


In [12]:
Console.WriteLine();
Console.WriteLine(new string('=', 70));
Console.WriteLine($"RESULT: {__passed} passed, {__failed.Count} failed.");
Console.WriteLine(new string('=', 70));

if (__failed.Count > 0)
{
    Console.WriteLine();
    Console.WriteLine("Failed assertions:");
    foreach (var f in __failed)
        Console.WriteLine($"  - {f}");
    throw new Exception($"{__failed.Count} SmartMapp.Net acceptance assertion(s) failed — see output above.");
}

Console.WriteLine();
Console.WriteLine("All Sprint 8 acceptance checks passed \u2714");



RESULT: 46 passed, 0 failed.

All Sprint 8 acceptance checks passed ✔
